In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../datasets/IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df.shape

(49582, 2)

## Text preprocessing

### 1. Converting to lowercase

In [6]:
df["review"] = df["review"].str.lower()

### 2. Removing the URL

In [7]:
import re #(regular expression)
def remove_urls(text):
    text = re.sub(r"http\S+","",text)  #(pattern,replacement,string) \S = nonwhitespace character (nospace)
    return text
    
df["review"] = df["review"].apply(remove_urls)

### 3. Remove Punctuations

In [8]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
    
df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML tag

In [9]:
def remove_htmltag(text):
    text = re.sub(r"<.*?>","",text)
    return text

df["review"] = df["review"].apply(remove_htmltag)

In [10]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5. Removing the stopwords

In [11]:
import nltk
nltk.download("punkt") # sentence tokanizer
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /home/saphal/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/saphal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/saphal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [13]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text
df["review"] = df["review"].apply(remove_stopwords)

### 6.Stemming

In [14]:
from nltk.stem import PorterStemmer

In [15]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"] = df["review"].apply(stemming)

In [16]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [17]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])
y = df["sentiment"]

In [18]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features = 5000)
X = tf.fit_transform(df["review"])

## Dataset and Dataloaders

In [20]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size = 0.2,random_state = 42
)

In [27]:
print(type(X_train))
print(type(X_test))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [28]:
import torch
from torch.utils.data import TensorDataset,DataLoader

# X_train = X_train.toarray() #to numpy
# X_test = X_test.toarray()

In [29]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [30]:
train_loader = DataLoader(train_set,shuffle = True,batch_size=64)
test_loader = DataLoader(test_set,shuffle= True,batch_size=64)

## Build our RNN

In [37]:
import torch.nn as nn
import torch.optim as optim

In [42]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn  = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        #fully connected layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        #optional steps
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out, _ = self.rnn(x,h0)
        #1st value = hidden state of all the timesteps ==> (batch,seq_len,hidden size)
        #2nd value = final hidden state of last timestep
        out = self.fc(out[:,-1,:])
        return out

In [43]:
input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training RNN